# Chapter 14: Rerankers and Cross-Encoders

Candidate-restricted reranking and the candidate ceiling.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
import hashlib
from document_identity import Collection
from evidence_units import build_units
from evidence_pipeline import identity, method_candidates, candidate_union, fuse, rerank, document_metrics

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
units = build_units(collection, "part2", "sections")
a = next(u for u in units if u["source_id"] == "toy:blog")
b = next(u for u in units if u["source_id"] == "toy:spec")
c = next(u for u in units if u["document_id"] == "b-focused")

pool = fuse({
    "bm25": method_candidates([{**a, "score": 3.0}, {**c, "score": 2.0}], "bm25", "part2", "hand-test"),
    "dense": method_candidates([{**a, "score": 3.0}, {**b, "score": 2.0}], "dense", "part2", "hand-test"),
}, 60)

def envelopes(candidates, scores_by_id):
    out = {}
    for cand in candidates:
        uid = cand["unit"]["passage_id"]
        out[uid] = {
            "score": scores_by_id[uid],
            "model_fingerprint": "manual-pair-control",
            "query_sha256": hashlib.sha256(b"query").hexdigest(),
            "text_sha256": hashlib.sha256(cand["unit"]["text"].encode()).hexdigest(),
        }
    return out

ids = [x["unit"]["passage_id"] for x in pool]
before_metrics = document_metrics([a["document_id"]], pool)
print("before rr@3:", before_metrics["rr@3"])

# Reranker promotes a different candidate: aggregate reciprocal rank worsens.
scores = {uid: (10.0 if uid != a["passage_id"] else 0.0) for uid in ids}
ranked = rerank(pool, envelopes(pool, scores), "manual-pair-control", "query")
after_metrics = document_metrics([a["document_id"]], ranked)
print("after rr@3:", after_metrics["rr@3"])
assert {x["unit"]["passage_id"] for x in ranked} == set(ids)
assert after_metrics["rr@3"] < before_metrics["rr@3"]

In [ ]:
# Missing scores cannot invent a candidate.
try:
    rerank(pool, {}, "manual-pair-control", "query")
    raised = False
except ValueError:
    raised = True
assert raised
print("missing candidate score refused:", raised)

In [ ]:
st = _support.try_import("sentence_transformers")
print("sentence_transformers available:", st is not None)
print("Real cross-encoder run: not executed (optional, cached model only).")

## Observed versus not established
Observed: candidate-set invariance and aggregate worsening from reordering. Not established: general reranking effectiveness.